In [1]:
import psycopg2

connection = psycopg2.connect(database="nba",
                host="localhost",
                user="postgres",
                password="191081843Boston!",
                port="5432")
cursor = connection.cursor()

In [2]:
import pandas as pd
import numpy as np

def execute_query(query):
    df = pd.DataFrame()
    cursor.execute(query)
    column_names = [desc[0] for desc in cursor.description]
    results = cursor.fetchall()
    df = pd.DataFrame(results, columns=column_names)
    return df


# Create a list to store all the DataFrames
traditional_stats_df = []
team_stats_df = []
games_df = []

# Loop over the 10 years of data
for i in range(10):
    year = 2024-i
    table_name = "traditional_stats_{}".format(year)
    query = "SELECT * FROM {}".format(table_name)
    table_df = execute_query(query)
    table_df['year'] = year
    traditional_stats_df.append(table_df)

    team_table_name = "team_stats_{}".format(year)
    team_query = "SELECT * FROM {}".format(team_table_name)
    team_table_df = execute_query(team_query)
    team_table_df['year'] = year
    team_stats_df.append(team_table_df)

    game_table_name = "games_{}".format(year)
    game_query = "SELECT * FROM {}".format(game_table_name)
    game_table_df = execute_query(game_query)
    game_table_df['year'] = year
    game_table_df['winner'] = np.where(game_table_df['home_points'] > game_table_df['away_points'],game_table_df['home'],game_table_df['away'])
    games_df.append(game_table_df)


# Concatenate all the DataFrames into a single DataFrame
traditional_stats_df = pd.concat(traditional_stats_df,ignore_index=True)
team_stats_df = pd.concat(team_stats_df,ignore_index=True)
games_df = pd.concat(games_df,ignore_index=True)


In [3]:
# create a dictionary mapping team names to abbreviations
abbreviations = {
   'Atlanta Hawks': 'ATL',
   'Boston Celtics': 'BOS',
   'Brooklyn Nets': 'BKN',
   'Charlotte Hornets': 'CHA',
   'Chicago Bulls': 'CHI',
   'Cleveland Cavaliers': 'CLE',
   'Dallas Mavericks': 'DAL',
   'Denver Nuggets': 'DEN',
   'Detroit Pistons': 'DET',
   'Golden State Warriors': 'GSW',
   'Houston Rockets': 'HOU',
   'Indiana Pacers': 'IND',
   'Los Angeles Clippers': 'LAC',
   'Los Angeles Lakers': 'LAL',
   'Memphis Grizzlies': 'MEM',
   'Miami Heat': 'MIA',
   'Milwaukee Bucks': 'MIL',
   'Minnesota Timberwolves': 'MIN',
   'New Orleans Pelicans': 'NOP',
   'New York Knicks': 'NYK',
   'Oklahoma City Thunder': 'OKC',
   'Orlando Magic': 'ORL',
   'Philadelphia 76ers': 'PHI',
   'Phoenix Suns': 'PHX',
   'Portland Trail Blazers': 'POR',
   'Sacramento Kings': 'SAC',
   'San Antonio Spurs': 'SAS',
   'Toronto Raptors': 'TOR',
   'Utah Jazz': 'UTA',
   'Washington Wizards': 'WAS'
}

# replace team names with abbreviations
team_stats_df['team'] = team_stats_df['team'].str.rstrip('*')
team_stats_df['team'] = team_stats_df['team'].replace(abbreviations)
games_df['home'] = games_df['home'].replace(abbreviations) 
games_df['away'] = games_df['away'].replace(abbreviations)
games_df['winner'] = games_df['winner'].replace(abbreviations)

In [ ]:
team_stats_df.head()

In [5]:
from bs4 import BeautifulSoup
import requests
import time

def get_games(year):
    months = ['october','november','december','january','february','march','april','may','june','july','august','september']
    ret = []
    for month in months:
        url = "https://www.basketball-reference.com/leagues/NBA_{}_games-{}.html".format(year,month)
        response = requests.get(url)
        time.sleep(3)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            links = soup.find_all('a')
            hrefs = [l.get('href') for l in links]
            box_scores = [f"https://www.basketball-reference.com{l}" for l in hrefs if l and "boxscore" in l and '.html' in l]
            ret.append(box_scores)
        else:
            print("bad",response.status_code)
            continue

    
    return ret

def extract_team_stats(team_stats_tag):
    team_stats = []
    rows = team_stats_tag.find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        cols = [ele.text.strip() for ele in cols]
        team_stats.append([ele for ele in cols if ele])
    
    return team_stats[0]

def process_game(url):
    response = requests.get(url)
    time.sleep(3)

    current_game = []
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        tables = soup.find_all('table', id=lambda x: x and 'game' in x)
        current_game = extract_team_stats(tables[0].find_all("tfoot")[0]) + extract_team_stats(tables[2].find_all("tfoot")[0])
        home,away = process_team_names(url)
        current_game.append(home)
        current_game.append(away)
        current_game.append(process_date(url))
        return current_game
    else:
        return response.status_code

def process_date(url):
    date = url.split("/")[4][0:8]
    return date

def process_team_names(url):
    response = requests.get(url)
    time.sleep(3)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        tables = soup.find_all('table', id=lambda x: x and 'game' in x)
        home_name = tables[0]['id'].split('-')[1]
        away_name = tables[2]['id'].split('-')[1]
        return home_name, away_name
    else:
        return response.status_code
    

def process_season(season):
    total_games = []
    for month in season:
        for url in month:
            current_game = process_game(url)
            total_games.append(current_game)
    return total_games



In [ ]:
games = []
year = 2024
for i in range(10):
    current = get_games(year)
    year -= 1
    games.append(current)

In [ ]:
box_scores = []
for i in range(10):
    current_season = games[i]
    box_score = process_season(current_season)
    box_scores.append(box_score)
    

In [ ]:
new_box_scores = []
for i in range(len(box_scores)):
    for j in range(len(box_scores[i])):
        if type(box_scores[i][j]) != list:
            print(box_scores[i][j])
            continue
        new_box_scores.append(box_scores[i][j][0:41]+[2024-i])

In [ ]:
columns = ["AWAY_MP","AWAY_FG","AWAY_FGA","AWAY_FG%","AWAY_3P","AWAY_3PA","AWAY_3P%","AWAY_FT","AWAY_FTA","AWAY_FT%","AWAY_ORB","AWAY_DRB","AWAY_TRB","AWAY_AST","AWAY_STL","AWAY_BLK","AWAY_TO","AWAY_PF","AWAY_PTS","HOME_MP","HOME_FG","HOME_FGA","HOME_FG%","HOME_3P","HOME_3PA","HOME_3P%","HOME_FT","HOME_FTA","HOME_FT%","HOME_ORB","HOME_DRB","HOME_TRB","HOME_AST","HOME_STL","HOME_BLK","HOME_TO","HOME_PF","HOME_PTS","AWAY","HOME","DATE","SEASON"]
box_scores_df = pd.DataFrame(new_box_scores, columns = columns)

In [ ]:
box_scores_df.to_csv("box_scores.csv")

In [ ]:
import pandas as pd

box_scores_df = pd.read_csv("box_scores.csv")
box_scores_df = box_scores_df.sort_values("DATE")
box_scores_df.head()

In [8]:
traditional_stats_df.head()
traditional_stats_df["total_minutes"] = traditional_stats_df["mp"] * traditional_stats_df["g"]

In [ ]:
top_players = traditional_stats_df.groupby(['team','year']).apply(
    lambda x: x.sort_values(by='total_minutes', ascending=False).head(8)
).reset_index(drop=True)
top_players.head()

In [10]:
#elo rating, win streak
import math

def win_probs(home_elo, away_elo, home_court_advantage) :
  h = math.pow(10, home_elo/400)
  r = math.pow(10, away_elo/400)
  a = math.pow(10, home_court_advantage/400) 

  denom = r + a*h
  home_prob = a*h / denom
  away_prob = r / denom 
  
  return home_prob, away_prob

def elo_k(MOV, elo_diff):
  k = 20
  if MOV>0:
      multiplier=(MOV+3)**(0.8)/(7.5+0.006*(elo_diff))
  else:
      multiplier=(-MOV+3)**(0.8)/(7.5+0.006*(-elo_diff))
  return k*multiplier

def update_elo(home_score, away_score, home_elo, away_elo, home_court_advantage) :
  home_prob, away_prob = win_probs(home_elo, away_elo, home_court_advantage) 

  if (home_score - away_score > 0) :
    home_win = 1 
    away_win = 0 
  else :
    home_win = 0 
    away_win = 1 
  
  k = elo_k(home_score - away_score, home_elo - away_elo)

  updated_home_elo = home_elo + k * (home_win - home_prob) 
  updated_away_elo = away_elo + k * (away_win - away_prob)

  return updated_home_elo, updated_away_elo

def get_prev_elo(team, date, season, box_scores, elo_df) :
  prev_game = box_scores[box_scores['DATE'] < date][(box_scores['HOME'] == team) | (box_scores['AWAY'] == team)].sort_values(by = 'DATE').tail(1).iloc[0]

  if team == prev_game['HOME'] :
    elo_rating = elo_df[elo_df['DATE'] == prev_game['DATE']][(elo_df["HOME"]==team) | (elo_df["AWAY"]==team)]['HOME_AFTER'].values[0]
  else :
    elo_rating = elo_df[elo_df['DATE'] == prev_game['DATE']][(elo_df["HOME"]==team) | (elo_df["AWAY"]==team)]['AWAY_AFTER'].values[0]

  
  if prev_game['SEASON'] != season :
    return (0.75 * elo_rating) + (0.25 * 1505)
  else :
    return elo_rating


In [ ]:
elo_df = pd.DataFrame(columns=['DATE', 'HOME', 'AWAY', 'HOME_BEFORE', 'AWAY_BEFORE', 'HOME_AFTER', 'AWAY_AFTER','SEASON'])

for index,row in box_scores_df.iterrows():
  game_date = row["DATE"]
  season = row['SEASON']
  home, away = row["HOME"], row["AWAY"]
  home_pts, away_pts = row["HOME_PTS"], row["AWAY_PTS"]
  
  if home not in elo_df['HOME'].values and home not in elo_df['AWAY'].values:
    home_elo_before = 1500
  else:
    home_elo_before = get_prev_elo(home, game_date, season, box_scores_df, elo_df)
      
  if away not in elo_df['HOME'].values and away not in elo_df['AWAY'].values:
    away_elo_before = 1500
  else:
    away_elo_before = get_prev_elo(away, game_date, season, box_scores_df, elo_df)

  home_elo_after, away_elo_after = update_elo(home_pts, away_pts, home_elo_before, away_elo_before, 69)

  new_row = {"DATE": game_date, "HOME": home, "AWAY": away, "HOME_BEFORE": home_elo_before, "AWAY_BEFORE": away_elo_before, "HOME_AFTER": home_elo_after, "AWAY_AFTER": away_elo_after,'SEASON':season}

  new_row_df = pd.DataFrame([new_row], columns=['DATE', 'HOME', 'AWAY', 'HOME_BEFORE', 'AWAY_BEFORE', 'HOME_AFTER', 'AWAY_AFTER','SEASON'])
  elo_df = pd.concat([elo_df, new_row_df], ignore_index=True)


In [ ]:
win_streak_df = pd.DataFrame(columns=["HOME", "AWAY", "DATE", "HOME_STREAK", "AWAY_STREAK"])



for index,row in box_scores_df.iterrows():
    home, away = row["HOME"], row["AWAY"]
    home_pts, away_pts = row["HOME_PTS"], row["AWAY_PTS"]
    date = row["DATE"]
    season = row["SEASON"]


    if home not in win_streak_df["HOME"].values and home not in win_streak_df["AWAY"].values:
        home_streak = 0
    else:
        prev_game = box_scores_df[box_scores_df["DATE"]<date][(box_scores_df["HOME"]==home) | (box_scores_df["AWAY"]==home)].sort_values(by="DATE").tail(1).iloc[0]
        if prev_game["SEASON"] != season:
            home_streak = 0
        else:
            if prev_game["HOME"]==home:
                if prev_game["HOME_PTS"] > prev_game["AWAY_PTS"]:
                    home_streak = win_streak_df[win_streak_df['DATE'] == prev_game['DATE']][win_streak_df["HOME"]==home]['HOME_STREAK'].values[0] + 1
                else:
                    home_streak = 0
            else:
                if prev_game["HOME_PTS"] < prev_game["AWAY_PTS"]:
                    home_streak = win_streak_df[win_streak_df['DATE'] == prev_game['DATE']][win_streak_df["AWAY"]==home]['AWAY_STREAK'].values[0] + 1
                else:
                    home_streak = 0

    if away not in win_streak_df["HOME"].values and away not in win_streak_df["AWAY"].values:
        away_streak = 0
    else:
        prev_game = box_scores_df[box_scores_df["DATE"]<date][(box_scores_df["HOME"]==away) | (box_scores_df["AWAY"]==away)].sort_values(by="DATE").tail(1).iloc[0]
        if prev_game["SEASON"] != season:
            away_streak = 0
        else:
            if prev_game["HOME"]==away:
                if prev_game["HOME_PTS"] > prev_game["AWAY_PTS"]:
                    away_streak = win_streak_df[win_streak_df['DATE'] == prev_game['DATE']][win_streak_df["HOME"]==away]['HOME_STREAK'].values[0] + 1
                else:
                    away_streak = 0
            else:
                if prev_game["HOME_PTS"] < prev_game["AWAY_PTS"]:
                    away_streak = win_streak_df[win_streak_df['DATE'] == prev_game['DATE']][win_streak_df["AWAY"]==away]['AWAY_STREAK'].values[0] + 1
                else:
                    away_streak = 0

    new_row = {"HOME":home, "AWAY":away, "DATE":date, "HOME_STREAK":home_streak, "AWAY_STREAK":away_streak}
    new_row_df = pd.DataFrame([new_row],columns=["HOME", "AWAY", "DATE", "HOME_STREAK", "AWAY_STREAK"])
    win_streak_df = pd.concat([new_row_df,win_streak_df],ignore_index=True)






In [13]:
from datetime import datetime, timedelta

def are_back_to_back_days(date1, date2):
    date1_dt = datetime.strptime(str(date1), "%Y%m%d")
    date2_dt = datetime.strptime(str(date2), "%Y%m%d")
    # Ensure that date2 is the next day after date1
    if date2_dt - date1_dt != timedelta(days=1):
        return 0

    # Check for edge cases involving month and year transitions
    if date1_dt.month != date2_dt.month or date1_dt.year != date2_dt.year:
        # Check for the case where date2 is the first day of the next month
        if date2_dt.day == 1 and date1_dt + timedelta(days=1) != date2_dt:
            return 0

        # Check for the case where date1 is the last day of a month and date2 is the first day of the same month in a leap year
        if date1_dt.day == datetime(date1_dt.year, date1_dt.month, 1) - timedelta(days=1) and date2_dt.day == 1 and date1_dt.year % 4 == 0 and (date1_dt.year % 100 != 0 or date1_dt.year % 400 == 0):
            return 0

    return 1

In [ ]:
b2b_df = pd.DataFrame(columns=["HOME","AWAY","DATE","HOME_B2B","AWAY_B2B"])

for index,row in box_scores_df.iterrows():
    home,away = row["HOME"], row["AWAY"]
    date = row["DATE"]
    season = row["SEASON"]
    
    if home not in b2b_df["HOME"].values and home not in b2b_df["AWAY"].values:
        home_b2b,away_b2b = 0,0
    else:
        home_prev_game = box_scores_df[box_scores_df["DATE"]<date][(box_scores_df["HOME"]==home) | (box_scores_df["AWAY"]==home)].sort_values(by="DATE").tail(1).iloc[0]
        away_prev_game = box_scores_df[box_scores_df["DATE"]<date][(box_scores_df["HOME"]==away) | (box_scores_df["AWAY"]==away)].sort_values(by="DATE").tail(1).iloc[0]

        home_b2b,away_b2b = are_back_to_back_days(home_prev_game["DATE"],date), are_back_to_back_days(away_prev_game["DATE"],date)

    new_row = {"HOME":home,"AWAY":away,"DATE":date,"HOME_B2B":home_b2b,"AWAY_B2B":away_b2b}
    new_row_df = pd.DataFrame([new_row],columns=["HOME","AWAY","DATE","HOME_B2B","AWAY_B2B"])
    b2b_df = pd.concat([b2b_df,new_row_df],ignore_index=True)


In [15]:
h2h_df = pd.DataFrame(columns=["HOME","AWAY","HOME_REC","AWAY_REC","DATE"])

for season in box_scores_df["SEASON"].unique():
    season_data = box_scores_df[box_scores_df["SEASON"] == season]
    matchups = {}
    for index,row in season_data.iterrows():
        home,away=row["HOME"],row["AWAY"]
        home_pts,away_pts = row["HOME_PTS"],row["AWAY_PTS"]
        date = row["DATE"]

        if home not in matchups:
            home_wins = 1 if home_pts > away_pts else 0
            total = 1
            matchups[home] = {away:[home_wins,total]}
        else:
            home_matchups = matchups[home]
            if away not in home_matchups:
                home_wins = 1 if home_pts > away_pts else 0
                total = 1
                home_matchups[away] = [home_wins,total]
            else:
                home_matchups[away][0] = home_matchups[away][0] + 1 if home_pts > away_pts else home_matchups[away][0]
                home_matchups[away][1] += 1

        if away not in matchups:
            away_wins = 1 if home_pts < away_pts else 0
            total = 1
            matchups[away] = {home:[away_wins,total]}
        else:
            away_matchups = matchups[away]
            if home not in away_matchups:
                away_wins = 1 if home_pts < away_pts else 0
                total = 1
                away_matchups[home] = [away_wins,total]
            else:
                away_matchups[home][0] = away_matchups[home][0] + 1 if home_pts < away_pts else away_matchups[home][0]
                away_matchups[home][1] += 1

        home_rec,away_rec = matchups[home][away][0]/matchups[home][away][1],matchups[away][home][0]/matchups[away][home][1]
        
        new_row = {"HOME":home,"AWAY":away,"HOME_REC":home_rec,"AWAY_REC":away_rec,"DATE":date}
        new_row_df = pd.DataFrame([new_row],columns=["HOME","AWAY","HOME_REC","AWAY_REC","DATE"])
        h2h_df = pd.concat([h2h_df,new_row_df],ignore_index=True)

In [ ]:
win_streak_df = win_streak_df.iloc[::-1].reset_index(drop=True)
win_streak_df.head()

In [ ]:
box_scores_df = box_scores_df.reset_index(drop=True)
games_df = pd.concat([box_scores_df, elo_df[["HOME_AFTER","AWAY_AFTER"]], win_streak_df[["HOME_STREAK","AWAY_STREAK"]], b2b_df[["HOME_B2B","AWAY_B2B"]], h2h_df[["HOME_REC","AWAY_REC"]]], axis=1)
games_df = games_df.drop('Unnamed: 0',axis=1)
games_df.head()

In [ ]:
label = [1 if row["HOME_PTS"] > row["AWAY_PTS"] else 0 for index,row in games_df.iterrows()]
essentials_df = games_df.loc[:, ["HOME","AWAY","HOME_AFTER","AWAY_AFTER","HOME_STREAK","AWAY_STREAK","HOME_B2B","AWAY_B2B","HOME_REC","AWAY_REC"]]
essentials_df['LABEL']=label
essentials_df.head()

In [19]:
essentials_df.to_csv("games.csv") 

In [20]:
games_df.to_csv("full_games.csv")